In [161]:
import os
from pathlib  import Path

In [162]:
os.chdir("../")
print(os.getcwd())

E:\ai


In [ ]:
import os

os.chdir(r"E:\ai\deep learning project")

print("Current Directory:", os.getcwd())

Current Directory: E:\ai\deep learning project


In [164]:
print(Path("config/config.yaml").exists())
print(Path("params.yaml").exists())

True
True


In [165]:
from dataclasses import dataclass  ##entity
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path



In [166]:
'''import yaml

with open("params.yaml", "r") as f:
    content = yaml.safe_load(f)

print(content)
print(type(content))'''

'import yaml\n\nwith open("params.yaml", "r") as f:\n    content = yaml.safe_load(f)\n\nprint(content)\nprint(type(content))'

In [167]:
from pathlib import Path

config = ConfigurationManager(
    config_filepath=Path("config/config.yaml"),
    params_filepath=Path("params.yaml")
)

print("Configuration loaded successfully!")

[2026-07-30 23:51:31,043: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-30 23:51:31,053: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-30 23:51:31,053: INFO: common: created directory at: artifacts]
Configuration loaded successfully!


In [168]:
from Cnnclassifier.constants import *
from Cnnclassifier.utils.common import read_yaml, create_directories

In [169]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
                


In [170]:
import os
import zipfile
import gdown
from Cnnclassifier import logger
from Cnnclassifier.utils.common import get_size

In [172]:
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
    def download_file(self):
        try:
            dataset_url = self.config.source_url
            zip_download_dir = self.config.local_data_file

            os.makedirs(self.config.root_dir, exist_ok=True)

            logger.info(f"Downloading data from {dataset_url}")

            file_id = dataset_url.split("/")[-2]

            prefix = "https://drive.google.com/uc?export=download&id="

            gdown.download(
                prefix + file_id,
                str(zip_download_dir),
                quiet=False
            )

            logger.info("Download completed successfully!")

        except Exception as e:
            raise e
    

    def extract_zip_file(self):
     unzip_path = self.config.unzip_dir

     os.makedirs(unzip_path, exist_ok=True)

     with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
         zip_ref.extractall(unzip_path)

    logger.info("Extraction completed.")

[2026-07-30 23:51:52,983: INFO: 952514837: Extraction completed.]


In [173]:
config = ConfigurationManager()

print("Configuration Loaded Successfully")

[2026-07-30 23:51:57,137: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-30 23:51:57,140: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-30 23:51:57,143: INFO: common: created directory at: artifacts]
Configuration Loaded Successfully


In [174]:
data_ingestion_config = config.get_data_ingestion_config()

print(data_ingestion_config)

[2026-07-30 23:52:00,451: INFO: common: created directory at: artifacts/data_ingestion]
DataIngestionConfig(root_dir='artifacts/data_ingestion', source_url='https://drive.google.com/file/d/1pwWJdrv-YYVsKYevXdn_zYhbbNzxhN6X/view?usp=sharing', local_data_file='artifacts/data_ingestion/data.zip', unzip_dir='artifacts/data_ingestion')


In [175]:
data_ingestion = DataIngestion(data_ingestion_config)
print("DataIngestion object created")

DataIngestion object created


In [176]:
data_ingestion.download_file()

[2026-07-30 23:52:07,948: INFO: 952514837: Downloading data from https://drive.google.com/file/d/1pwWJdrv-YYVsKYevXdn_zYhbbNzxhN6X/view?usp=sharing]


Downloading...
From (original): https://drive.google.com/uc?id=1pwWJdrv-YYVsKYevXdn_zYhbbNzxhN6X
From (redirected): https://drive.google.com/uc?id=1pwWJdrv-YYVsKYevXdn_zYhbbNzxhN6X&confirm=t&uuid=0e65ab57-ec03-456f-a92d-fc53cad1d751
To: E:\ai\deep learning project\artifacts\data_ingestion\data.zip
100%|██████████| 1.67G/1.67G [01:11<00:00, 23.5MB/s]

[2026-07-30 23:53:23,098: INFO: 952514837: Download completed successfully!]


In [177]:
data_ingestion.extract_zip_file()